In [26]:
# Importações
import pandas as pd
import re
import numpy as np
import emoji, re, string, time, os
import spacy
import nltk
from nltk.corpus import stopwords

import matplotlib.pyplot as plt 
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import BernoulliNB, MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import SGDClassifier

from sklearn.model_selection import cross_validate
from scipy.sparse import hstack, csr_matrix
from sklearn.model_selection import train_test_split, cross_validate, learning_curve, StratifiedKFold
from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay, roc_auc_score,
    accuracy_score, precision_score, recall_score, f1_score
)
import time

from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import CountVectorizer

In [27]:
# Caminhos para o dataset e dicionário
dataset_path = "../../Datasets/Correto_whatsapp_rotulado_revisado.csv"
dic_path = "../../Dicionário/v2_SocialLIWC_formatado_ordenado.dic"

# Carrega o dataset
df = pd.read_csv(dataset_path)
df['text_content_anonymous'] = df['text_content_anonymous'].astype(str)

### Carrega o PrejudiceBR

In [28]:
# Função para carregar o dicionário
def carregar_dicionario_personalizado(dic_path):
    categorias = {}
    lexicon = {}
    dentro_das_categorias = False

    with open(dic_path, 'r', encoding='utf-8') as file:
        for linha in file:
            linha = linha.strip()
            
            # Detecta a seção de categorias delimitada por '%'
            if linha == '%':
                dentro_das_categorias = not dentro_das_categorias
                continue
            
            # Lê as categorias personalizadas
            if dentro_das_categorias:
                codigo, categoria = linha.split()
                categorias[codigo] = categoria
            else:
                # Lê as palavras e suas categorias
                partes = linha.split("\t")
                palavra = partes[0]
                categoria_ids = partes[1:]
                lexicon[palavra] = [categorias[codigo] for codigo in categoria_ids if codigo in categorias]
    
    return lexicon, list(categorias.values())

### Tokenizar e organizar as palavras como features

In [29]:
# Tokenização simples
def tokenize(text):
    tokens = []
    for match in re.finditer(r"\w+", text, re.UNICODE):
        tokens.append(match.group(0).lower())
    return tokens

# Função para incluir palavras do dicionário como colunas
def adicionar_palavras_como_features(df, lexicon):
    # Criar DataFrame temporário com todas as palavras inicializadas com 0
    new_columns = pd.DataFrame(0, index=df.index, columns=list(lexicon.keys()))
    
    # Concatenar com o DataFrame original de uma vez
    df = pd.concat([df, new_columns], axis=1)
    
    # Contar ocorrências de cada palavra no texto
    for i, texto in df['text_content_anonymous'].items():
        tokens = tokenize(texto)
        for token in tokens:
            if token in lexicon:
                df.at[i, token] += 1
                
    return df

In [30]:
experiments = [
               'ml-tfidf-uni_gram',
               'ml-tfidf-uni_gram-processed',
               'ml-tfidf-unibi_gram',
               'ml-tfidf-unibi_gram-processed',
               'ml-tfidf-unibitri_gram',
               'ml-tfidf-unibitri_gram-processed',

            
               'ml-bow-uni_gram',
               'ml-bow-uni_gram-processed',
               'ml-bow-unibi_gram',
               'ml-bow-unibi_gram-processed',
               'ml-bow-unibitri_gram',
               'ml-bow-unibitri_gram-processed'
    
]

### Configuração do baseline

In [31]:
unicode_emoji = {}
for key, value in emoji.EMOJI_DATA.items():
    try:
        unicode_emoji[key] = value['pt']
    except:
        pass

#emojis and punctuation
emojis_list = list(unicode_emoji)
punct = list(string.punctuation)
emojis_punct = emojis_list + punct

def processEmojisPunctuation(text, remove_punct = True):
    '''
    Put spaces between emojis. Removes punctuation.
    '''
    #get all unique chars
    chars = set(text)
    #for each unique char in text, do:
    for c in chars:
        #remove punctuation
        if remove_punct:
            if c in emojis_list:
                text = text.replace(c, ' ' + c + ' ')
            if c in punct:
                text = text.replace(c, ' ')

        #put spaces between punctuation
        else:
            if c in emojis_punct:
                text = text.replace(c, ' ' + c + ' ')          

    text = text.replace('  ', ' ')
    return text

#stop words removal
stop_words = list(stopwords.words('portuguese'))
new_stopwords = ['aí','pra','vão','vou','onde','lá','aqui',
                 'tá','pode','pois','so','deu','agora','todo',
                 'nao','ja','vc', 'bom', 'ai','kkk','kkkk','ta', 'voce', 'alguem', 'ne', 'pq',
                 'cara','to','mim','la','vcs','tbm', 'tudo']
stop_words = stop_words + new_stopwords
final_stop_words = []
for sw in stop_words:
    sw = ' '+ sw + ' '
    final_stop_words.append(sw)

def removeStopwords(text):
    for sw in final_stop_words:
        text = text.replace(sw,' ')
    text = text.replace('  ',' ')
    return text

#lemmatization
nlp = spacy.load('pt_core_news_sm')
def lemmatization(text):
    doc = nlp(text)
    for token in doc:
        if token.text != token.lemma_:
            text = text.replace(token.text, token.lemma_)
    return text


def domainUrl(text):
    '''
    Substitutes an URL in a text for the domain of this URL
    Input: an string
    Output: the string with the modified URL
    '''    
    if 'http' in text:
        re_url = '[^\s]*https*://[^\s]*'
        matches = re.findall(re_url, text, flags=re.IGNORECASE)
        for m in matches:
            domain = m.split('//')
            domain = domain[1].split('/')[0]
            text = re.sub(re_url, domain, text, 1)
        return text
    else:
        return text 

def preprocess(text):
    text = text.lower().strip()
    text = domainUrl(text)
    text = processEmojisPunctuation(text)
    text = removeStopwords(text)
    text = lemmatization(text)
    return text


In [32]:
# Configuração do vetorizador com base no experimento
def configure_vectorizer(experiment):
    if 'tfidf' in experiment:
        if 'uni_gram' in experiment:
            return TfidfVectorizer(ngram_range=(1, 1))
        elif 'unibi_gram' in experiment:
            return TfidfVectorizer(ngram_range=(1, 2))
        elif 'unibitri_gram' in experiment:
            return TfidfVectorizer(ngram_range=(1, 3))
        else:
            return TfidfVectorizer()
    elif 'bow' in experiment:
        if 'uni_gram' in experiment:
            return CountVectorizer(binary=True, ngram_range=(1, 1))
        elif 'unibi_gram' in experiment:
            return CountVectorizer(binary=True, ngram_range=(1, 2))
        elif 'unibitri_gram' in experiment:
            return CountVectorizer(binary=True, ngram_range=(1, 3))
        else:
            return CountVectorizer(binary=True)


In [33]:
# Função de pré-processamento
def preprocess_data(df, experiment):
    if 'processed' in experiment:
        print("Pré-processamento ativado.")
        pro_texts = [preprocess(t) for t in df['text_content_anonymous']]
    else:
        print("Sem pré-processamento.")
        pro_texts = [processEmojisPunctuation(t.lower(), remove_punct=False) for t in df['text_content_anonymous']]
    return pro_texts


In [34]:
lexicon, category_names = carregar_dicionario_personalizado(dic_path)

# Rodar para cada experimento
for experiment in experiments:
    start_time = time.time()

    try:
        print(f"Rodando experimento: {experiment}")

        # Removendo duplicatas e lidando com valores ausentes
        df = df.drop_duplicates(subset=['text_content_anonymous'])
        df['text_content_anonymous'] = df['text_content_anonymous'].fillna("")
        df['text_content_anonymous'] = df['text_content_anonymous'].astype(str)

        # Processar textos dependendo do experimento
        pro_texts = preprocess_data(df, experiment)

        # Configuração de vetorizador
        vectorizer = configure_vectorizer(experiment)
        X_texts = vectorizer.fit_transform(pro_texts)
        vocab_size_texts = X_texts.shape[1]

        # Printar o número de features do vetorizador
        print(f"[{experiment}] Número de features do vetorizador: {vocab_size_texts}")

        # --- DICIONÁRIO ---
        df_dic = adicionar_palavras_como_features(df.copy(), lexicon)
        X_dic = csr_matrix(df_dic[list(lexicon.keys())].values)

        # --- COMBINAR ---
        X_combined = hstack([X_texts, X_dic], format="csr")

        

        vocab_size_combined = X_combined.shape[1]

        y = df['preconceito']

        # Dividir os dados em treino e teste
        X_train, X_test, y_train, y_test = train_test_split(
            X_combined, y, test_size=0.2, stratify=y, random_state=42
        )

        # Definir os modelos a serem usados
        models = [
            LogisticRegression(), BernoulliNB(), MultinomialNB(), LinearSVC(dual=False),
            KNeighborsClassifier(), SGDClassifier(), RandomForestClassifier(),
            GradientBoostingClassifier(n_estimators=200),
            MLPClassifier(batch_size=64, max_iter=50, early_stopping=True, verbose=False)
        ]

     

        resultados = []

        # Avaliando os modelos
        for model in models:
            resultado_linha = dict()
            resultado_linha['model'] = model.__class__.__name__
            resultado_linha['vocab_size_combined'] = vocab_size_combined

            # Medir o tempo de execução
            model_start_time = time.time()
            metodos_scoring = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']
            cv_results = cross_validate(model, X_train, y_train, cv=5, scoring=metodos_scoring, return_train_score=True)
            model_end_time = time.time()
            execution_time = model_end_time - model_start_time

            resultado_linha['execution_time'] = execution_time

            # Salvar as métricas
            for scoring in metodos_scoring:
                scores = cv_results[f'test_{scoring}']
                resultado_linha[f'{scoring}_avg'] = scores.mean()
                resultado_linha[f'{scoring}_std'] = scores.std()

            resultados.append(resultado_linha)

            
            # Printar informações durante a execução
            print(f"Model: {model.__class__.__name__} | Experiment: {experiment}")
            print(f"Vocab Size (Combined): {vocab_size_combined}")
            print(f"Execution Time: {execution_time:.2f} seconds")
            print("-" * 50)

        # Tempo total do experimento
        experiment_time = time.time() - start_time
        print(f"Tempo total do experimento '{experiment}': {experiment_time:.2f} seconds")

        # Criar um DataFrame com os resultados
        df_metrics = pd.DataFrame(resultados)
        df_metrics['total_time'] = experiment_time  # Adicionando o tempo total

        # Salvar os resultados em um arquivo CSV
        path_to_save = f"17_10_25_WhatsApp_resultados_{experiment}.csv"
        df_metrics.to_csv(path_to_save, index=False)

        print(f"Resultados salvos em: {path_to_save}")

    except Exception as e:
        print(f"[{experiment}] Erro ao executar o experimento: {e}")


Rodando experimento: ml-tfidf-uni_gram
Sem pré-processamento.
[ml-tfidf-uni_gram] Número de features do vetorizador: 12539
Model: LogisticRegression | Experiment: ml-tfidf-uni_gram
Vocab Size (Combined): 13381
Execution Time: 0.88 seconds
--------------------------------------------------
Model: BernoulliNB | Experiment: ml-tfidf-uni_gram
Vocab Size (Combined): 13381
Execution Time: 0.10 seconds
--------------------------------------------------
Model: MultinomialNB | Experiment: ml-tfidf-uni_gram
Vocab Size (Combined): 13381
Execution Time: 0.08 seconds
--------------------------------------------------
Model: LinearSVC | Experiment: ml-tfidf-uni_gram
Vocab Size (Combined): 13381
Execution Time: 0.29 seconds
--------------------------------------------------
Model: KNeighborsClassifier | Experiment: ml-tfidf-uni_gram
Vocab Size (Combined): 13381
Execution Time: 4.88 seconds
--------------------------------------------------
Model: SGDClassifier | Experiment: ml-tfidf-uni_gram
Vocab Si